In [ ]:
!pip install sympy==1.11
!pip install torchmetrics

In [ ]:

import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms.functional as TF

# =====================================================================
# ENVIRONMENT & DIRECTORY CONFIGURATION
# =====================================================================
# Set local directory paths for tracking experiment logs and model checkpoints
project_path     = "./BEng_Project"
experiments_path = f"{project_path}/experiments"
os.makedirs(experiments_path, exist_ok=True)

# =====================================================================
# DATASET INITIALISATION
# =====================================================================
# Load cross-modal SHG microscopy image arrays from the local project directory
# Note: Raw .npy arrays are withheld from the repository to protect dataset privacy
try:
    backwards = np.load('datasets/backwards_small.npy')
    forwards  = np.load('datasets/forwards_small.npy')
    print(f"Dataset successfully loaded.\nBackward SHG Shape: {backwards.shape}\nForward SHG Shape: {forwards.shape}")
except FileNotFoundError:
    print("Dataset files not found. Please verify that 'backwards_small.npy' and 'forwards_small.npy' reside in a '/datasets' root folder.")

# =====================================================================
# PYTORCH DATA PIPELINE & AUGMENTATION
# =====================================================================
class AugmentedDataset(Dataset):
    """
    Custom PyTorch Dataset class to manage cross-modal SHG microscopy image matrices.
    Applies real-time stochastic spatial transformations to minimize network overfitting.
    """
    def __init__(self, images, masks, augment=True):
        self.images  = images
        self.masks   = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx]
        mask = self.masks[idx]

        # Apply spatial data augmentations with a 90% execution probability per transformation
        if self.augment:
            if random.random() > 0.1:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            if random.random() > 0.1:
                img  = TF.vflip(img)
                mask = TF.vflip(mask)

            # Stochastic rotation across 90-degree increments
            k = random.randint(0, 2)
            if k > 0:
                img  = torch.rot90(img,  k, dims=[1, 2])
                mask = torch.rot90(mask, k, dims=[1, 2])

        return img, mask


In [ ]:
# ############# DESIGNING Unet #############

import torch
import torch.nn as nn
import torch.nn.functional as F # Changed TF to F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False), # responsible for feature extraction. The bias is false as its cancelled by the instanc norm
            nn.InstanceNorm2d(out_channels), #InstanceNorm2d normalizes each image independently, making training stable for small bactch sizes
            nn.ReLU(inplace=True), # Introduces nonlinearity, which allows the model to learn complex relationships with out it the model would just be a linear model
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNET(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512, 1024]):
        super(UNET, self).__init__()
        self.ups = nn.ModuleList() #stores convolution layers
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.decoder_dropouts = nn.ModuleList([nn.Dropout2d(0.3) for _ in features ])
        self.bottleneck_dropout = nn.Dropout2d(0.3)

        # Downsampling path
        current_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(current_channels, feature))
            current_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Upsampling path
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2) # feature*2 as we are concatenating our feature through the skip conncetions
            )
            self.ups.append(DoubleConv(feature * 2, feature))

        # Final output layer
        #self.final_conv = nn.Sequential(
            #nn.Conv2d(features[0], out_channels, kernel_size=1),
            #nn.Sigmoid()
        #)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)


    def forward(self, x):
        skip_connections = []

        # Encoder
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)
        x = self.bottleneck_dropout(x) # dropout
        skip_connections = skip_connections[::-1] # the list is reversed to make it easier

        # Decoder
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            x = self.decoder_dropouts[idx // 2](x) # dropout
            skip_connection = skip_connections[idx // 2]

            # Fix shape mismatch if needed
            if x.shape[2:] != skip_connection.shape[2:]:
                x = F.interpolate(x,size=skip_connection.shape[2:],mode="bilinear",align_corners=False) # Changed TF.interpolate to F.interpolate
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx + 1](concat_skip)

        return self.final_conv(x)

In [ ]:
import torch #  torch operations for tensors
from skimage import exposure
from scipy.spatial import cKDTree

# ── Metrics ────────────────────────────
def compute_iou(pred, target, threshold=0.2):
    pred_bin = (pred > threshold).float()
    p = pred_bin.detach().cpu().numpy().flatten()
    t = target.detach().cpu().numpy().flatten()
    intersection = np.sum(p * t)
    union = np.sum(p) + np.sum(t) - intersection
    return float(intersection / (union + 1e-7))

def compute_dice(pred, target, threshold=0.2):
    pred_bin = (pred > threshold).float()
    p = pred_bin.detach().cpu().numpy().flatten()
    t = target.detach().cpu().numpy().flatten()
    intersection = np.sum(p * t)
    return float((2 * intersection + 1e-7) / (np.sum(p) + np.sum(t) + 1e-7))

def hausdorff_95(y_true, y_pred):
    """
    Lower is better (distance in pixels).
    """
    # Move tensors to CPU and convert to numpy arrays
    y_true_np = y_true.cpu().numpy() # shape (B, 1, H, W)
    y_pred_np = y_pred.cpu().numpy() # shape (B, 1, H, W)

    batch_size = y_true_np.shape[0]
    metric_values = []

    for i in range(batch_size):
        single_y_true = y_true_np[i, 0, :, :] # shape (H, W)
        single_y_pred = y_pred_np[i, 0, :, :] # shape (H, W)

        # Threshold for finding the points
        u = np.argwhere(single_y_true > 0.5)
        v = np.argwhere(single_y_pred > 0.5)

        if len(u) == 0 or len(v) == 0:
            metric_values.append(100.0) # Error fallback
            continue

        # Calculate distance to closest points
        dist_uv, _ = cKDTree(v).query(u)
        dist_vu, _ = cKDTree(u).query(v)

        metric_values.append(max(np.percentile(dist_uv, 95), np.percentile(dist_vu, 95)))
    return np.mean(metric_values) if metric_values else 100.0

def Blockwise_diff(target, pred, block_size):

    # Automatically handle channel dimension if present (e.g., (B, 1, H, W))
    if target.dim() == 4 and target.shape[1] == 1:
        target = target.squeeze(1)
    if pred.dim() == 4 and pred.shape[1] == 1:
        pred = pred.squeeze(1)

    # Now both target and pred should be (B, H, W)
    # Dimension check for (B, H, W) format
    if target.dim() != 3 or pred.dim() != 3:
        raise ValueError("Input tensors must have 3 dimensions (B, H, W) after optional squeezing.")

    if target.shape != pred.shape:
        raise ValueError("Target and prediction images must have the same dimensions (B, H, W).")

    bh, bw = block_size

    num_images, H, W = target.shape # Use num_images, H, W for simplicity after check

    # Initialize a list to store all average block differences
    all_avg_diffs = []

    for i in range(num_images):
        for y in range(0, H, bh):
            for x in range(0, W, bw):
                pred_b = pred[i, y : min(y + bh, H), x : min(x + bw, W)] # Ensure bounds
                target_b = target[i, y : min(y + bh, H), x : min(x + bw, W)] # Ensure bounds

                # Calculate block difference and its average absolute value
                diff = pred_b - target_b
                avg_block_diff = torch.mean(torch.abs(diff))

                # Collect the average difference
                all_avg_diffs.append(avg_block_diff)

    # Return the overall average of all collected block differences
    if all_avg_diffs:
        return torch.mean(torch.stack(all_avg_diffs))
    else:
        return torch.tensor(0.0, device=target.device)



In [ ]:
import os
import datetime
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# =====================================================================
# MODEL TRAINING ROUTINE
# =====================================================================
def train_fn(loader, model, optimizer, loss_fn, scaler, device):
    """
    Executes a single training epoch using Mixed Precision (AMP) to maximize throughput.
    """
    model.train()
    epoch_loss = 0.0

    for data, targets in tqdm(loader, desc="Training Batches", leave=False):
        data, targets = data.to(device), targets.to(device)

        # Forward pass with automatic mixed precision
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # Backward pass with gradient scaling
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

# =====================================================================
# MODEL VALIDATION & INFERENCE ROUTINE
# =====================================================================
def val_fn(loader, model, loss_fn, epoch, device, run_path, block_size):
    """
    Evaluates model performance across continuous and thresholded binary inference
    modalities, computing structural boundaries and displacement error metrics.
    """
    model.eval()
    val_loss = val_iou = val_dice = val_hausdorff_95 = val_blockwise_diff = 0.0

    with torch.no_grad():
        for idx, (data, targets) in enumerate(loader):
            data, targets = data.to(device), targets.to(device)
            predictions = model(data)

            val_loss += loss_fn(predictions, targets).item()

            # Generate dual inference paths
            preds_sig = torch.sigmoid(predictions)
            preds_binary = (preds_sig > 0.2).float() # Set target threshold mapping

            # Quantitative metrics evaluated on binarized predictions
            val_iou            += compute_iou(preds_binary, targets)
            val_dice           += compute_dice(preds_binary, targets)
            val_hausdorff_95   += hausdorff_95(preds_binary, targets)
            val_blockwise_diff += Blockwise_diff(targets, preds_binary, block_size).item()

            # Generate validation inference grid every 10 epochs
            if idx == 0 and (epoch % 10 == 0):
                fig, axes = plt.subplots(3, 4, figsize=(12, 9))
                for i in range(3):
                    axes[i,0].imshow(exposure.equalize_adapthist(data[i].squeeze(0).cpu().numpy(), clip_limit=0.018), cmap='viridis')
                    axes[i,0].set_title("Input"); axes[i,0].axis('off')

                    axes[i,1].imshow(exposure.equalize_adapthist(targets[i].squeeze(0).cpu().numpy(), clip_limit=0.01), cmap='gray')
                    axes[i,1].set_title("Ground Truth"); axes[i,1].axis('off')

                    axes[i,2].imshow(exposure.equalize_adapthist(preds_sig[i].squeeze(0).cpu().numpy(), clip_limit=0.01), cmap='gray')
                    axes[i,2].set_title("Prediction (Cont.)"); axes[i,2].axis('off')

                    axes[i,3].imshow(preds_binary[i].squeeze(0).cpu().numpy(), cmap='gray')
                    axes[i,3].set_title("Prediction (Binary)"); axes[i,3].axis('off')

                plt.suptitle(f"Inference Modality Comparison - Epoch {epoch+1}")
                plt.tight_layout()
                plt.savefig(f"{run_path}/predictions_epoch{epoch+1}.png", bbox_inches='tight')
                plt.close()

    n = len(loader)
    return val_loss/n, val_iou/n, val_dice/n, val_hausdorff_95/n, val_blockwise_diff/n

# =====================================================================
# HYPERPARAMETER & RUN INITIALISATION
# =====================================================================
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 0.001
num_epochs    = 150
batch_size    = 35
block_size    = (32, 32)

# Generate unique execution runtime path
run_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = f"UNet_bs{batch_size}_lr{learning_rate}_{run_time}"
run_path = f"{experiments_path}/{run_name}"
os.makedirs(run_path, exist_ok=True)

# Format tensor array input bounds
backward_tensor = torch.from_numpy(backwards).unsqueeze(1).float()
forward_tensor  = torch.from_numpy(forwards).unsqueeze(1).float()

# Execute stratified train/validation/test splits (80/10/10)
X_train, X_test, y_train, y_test = train_test_split(
    backward_tensor, forward_tensor, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42)

# Instantiate optimization data loaders
train_dataloader = DataLoader(AugmentedDataset(X_train, y_train, augment=True), batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

# =====================================================================
# NETWORK COMPONENT INSTANTIATION
# =====================================================================
model     = UNET(in_channels=1, out_channels=1).to(device)
loss_fn   = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler    = torch.cuda.amp.GradScaler()

train_losses = []
val_losses, val_ious, val_dices, val_hausdorff_95, val_blockwise_diff = [], [], [], [], []

# Generate execution manifest file
with open(f"{run_path}/run_manifest.txt", "w") as f:
    f.write(f"Model Architecture: {str(model)}\n\n")
    f.write(f"Learning Rate:    {learning_rate}\n")
    f.write(f"Batch Size:       {batch_size}\n")
    f.write(f"Total Epochs:     {num_epochs}\n")
    f.write(f"Data Augmentations: HFlip, VFlip, Rot90\n")
    f.write(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}\n")

# =====================================================================
# TRAINING & EVALUATION LOOP EXECUTION
# =====================================================================
for epoch in range(num_epochs):
    train_loss = train_fn(train_dataloader, model, optimizer, loss_fn, scaler, device)
    val_loss, val_iou, val_dice, val_hd95, val_diff = val_fn(
        val_dataloader, model, loss_fn, epoch, device, run_path, block_size)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ious.append(val_iou)
    val_dices.append(val_dice)
    val_hausdorff_95.append(val_hd95)
    val_blockwise_diff.append(val_diff)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {val_iou:.4f} | DICE: {val_dice:.4f}")

# Save final checkpoint weights
torch.save(model.state_dict(), f"{run_path}/model_checkpoint.pth")

# =====================================================================
# METRIC VISUALISATION PLOT GENERATION
# =====================================================================
epochs_range = range(1, num_epochs + 1)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("U-Net Training Performance Summary Metrics")

# Subplot 0: Loss Tracks
axes[0,0].plot(epochs_range, train_losses, label="Train")
axes[0,0].plot(epochs_range, val_losses, label="Validation")
axes[0,0].set_title("BCE Loss Curve"); axes[0,0].set_xlabel("Epoch"); axes[0,0].legend()

# Subplot 1: Jaccard Index tracking
axes[0,1].plot(epochs_range, val_ious, color="tab:orange")
axes[0,1].set_title("Validation IoU (Jaccard Index)"); axes[0,1].set_xlabel("Epoch")

# Subplot 2: Dice score tracking
axes[0,2].plot(epochs_range, val_dices, color="tab:green")
axes[0,2].set_title("Validation DICE Coefficient"); axes[0,2].set_xlabel("Epoch")

# Subplot 3: Hausdorff Distance Tracking
axes[1,0].plot(epochs_range, val_hausdorff_95, color="tab:blue")
axes[1,0].set_title("95th Percentile Hausdorff Distance (HD95)"); axes[1,0].set_xlabel("Epoch")

# Subplot 4: Blockwise structural differences
axes[1,1].plot(epochs_range, val_blockwise_diff, color="tab:gray")
axes[1,1].set_title("Validation Blockwise Variance Difference"); axes[1,1].set_xlabel("Epoch")

# Subplot 5: Blank layout mapping balance
axes[1,2].axis('off')

plt.tight_layout()
plt.savefig(f"{run_path}/training_summary_plot.png", dpi=150)
plt.close()

# Save final execution telemetry arrays
metrics_map = {
    "train_losses": train_losses, "val_losses": val_losses,
    "val_ious": val_ious, "val_dices": val_dices,
    "val_hausdorff_95": val_hausdorff_95, "val_blockwise_diff": val_blockwise_diff
}
for name, array in metrics_map.items():
    np.save(f"{run_path}/{name}.npy", np.array(array))

# Append final performance statistics summaries
with open(f"{run_path}/run_manifest.txt", "a") as f:
    f.write(f"\n=====================================================================\n")
    f.write(f"FINAL EVALUATION METRICS (HELD-OUT TEST SET):\n")
    f.write(f"=====================================================================\n")
    f.write(f"  IoU:           {val_ious[-1]:.6f} (± {np.std(val_ious):.6f})\n")
    f.write(f"  DICE:          {val_dices[-1]:.6f} (± {np.std(val_dices):.6f})\n")
    f.write(f"  HD95 Error:    {val_hausdorff_95[-1]:.6f}px (± {np.std(val_hausdorff_95):.6f}px)\n")
    f.write(f"  Blockwise Δ:   {val_blockwise_diff[-1]:.6f} (± {np.std(val_blockwise_diff):.6f})\n")


In [ ]:
import os
import tifffile
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# =====================================================================
# MODEL SPECIFICATION & LOAD ROUTINE
# =====================================================================
# Option A: Inherit the operational model directly from memory (Cell 2/3 execution)
# run_path, device, and model variables are already instantiated.

# Option B: Reload a checkpoint configuration from local storage
# uncomment lines below to parse a previous run.
# model_path = "./BEng_Project/experiments/UNet_bs35_lr0.001_xxxxxx/model_checkpoint.pth"
# device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# run_path   = os.path.dirname(model_path)
# model      = UNET(in_channels=1, out_channels=1).to(device)
# model.load_state_dict(torch.load(model_path, map_location=device))

model.eval()
print(f"Executing inference. Saving test artifacts to local directory: {run_path}")

# =====================================================================
# TEST PIPELINE RECONSTRUCTION
# =====================================================================
# Ensure tensors use exact matching random states to isolate the target test data partition
backward_tensor = torch.from_numpy(backwards).unsqueeze(1).float()
forward_tensor  = torch.from_numpy(forwards).unsqueeze(1).float()

_, X_test, _, y_test = train_test_split(
    backward_tensor, forward_tensor, test_size=0.1, random_state=42)

test_dataloader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=35, shuffle=False)

# =====================================================================
# METRIC EVALUATION LOOP INITIALISATION
# =====================================================================
all_inputs, all_preds, all_preds_bi, all_targets = [], [], [], []
iou_scores, dice_scores, hausdorff_95_scores = [], [], []

block_size = (32, 32)
all_blockwise_diffs = []

# =====================================================================
# ON-TEST BATCH INFERENCE EVALUATION
# =====================================================================
with torch.no_grad():
    for data, targets in test_dataloader:
        data, targets = data.to(device), targets.to(device)

        # Dual inference map generation
        preds_sig = torch.sigmoid(model(data))
        preds_binary = (preds_sig > 0.2).float()

        # Compile metric tracks across batches
        iou_scores.append(compute_iou(preds_sig, targets))
        dice_scores.append(compute_dice(preds_sig, targets))
        hausdorff_95_scores.append(hausdorff_95(preds_binary, targets))

        # Blockwise variance matrix execution
        avg_diff = Blockwise_diff(targets, preds_binary, block_size)
        all_blockwise_diffs.append(avg_diff.item())

        # Collect tracking data to local host storage
        all_inputs.append(data.cpu())
        all_preds.append(preds_sig.cpu())
        all_preds_bi.append(preds_binary.cpu())
        all_targets.append(targets.cpu())

# Flatten tensor matrix outputs for array formatting
all_inputs   = torch.cat(all_inputs,   dim=0)[:, 0].numpy().astype(np.float32)
all_preds    = torch.cat(all_preds,    dim=0)[:, 0].numpy().astype(np.float32)
all_preds_bi = torch.cat(all_preds_bi, dim=0)[:, 0].numpy().astype(np.float32)
all_targets  = torch.cat(all_targets,  dim=0)[:, 0].numpy().astype(np.float32)

# =====================================================================
# ARTIFACT & MULTI-PAGE TIFF MATRIX EXPORT
# =====================================================================
tifffile.imwrite(f"{run_path}/test_inputs.tif",          all_inputs)
tifffile.imwrite(f"{run_path}/test_predictions.tif",     all_preds)
tifffile.imwrite(f"{run_path}/test_predictions_bi.tif",  all_preds_bi)
tifffile.imwrite(f"{run_path}/test_ground_truth.tif",    all_targets)

# Compute unified telemetry statistical values
test_iou          = float(np.mean(iou_scores))
test_dice         = float(np.mean(dice_scores))
test_hausdorff_95 = float(np.mean(hausdorff_95_scores))
overall_avg_blockwise_diff = float(np.mean(all_blockblockwise_diffs))

print(f"\nSuccessfully generated and exported {len(all_preds)} image slices.")
print(f"---------------------------------------------------------------------")
print(f"Test IoU:           {test_iou:.4f} (± {np.std(iou_scores):.4f})")
print(f"Test DICE:          {test_dice:.4f} (± {np.std(dice_scores):.4f})")
print(f"Test Hausdorff_95:  {test_hausdorff_95:.4f}px (± {np.std(hausdorff_95_scores):.4f}px)")
print(f"Blockwise Variance: {overall_avg_blockwise_diff:.4f} (± {np.std(all_blockwise_diffs):.4f})")

# =====================================================================
# EXPORT LOG MANIFEST GENERATION
# =====================================================================
with open(f"{run_path}/run_manifest.txt", "a") as f:
    f.write(f"\n=====================================================================\n")
    f.write(f"HELD-OUT EVALUATION METRICS SUMMARY RESULTS:\n")
    f.write(f"=====================================================================\n")
    f.write(f"  Final IoU:            {test_iou:.6f} (± {np.std(iou_scores):.6f})\n")
    f.write(f"  Final DICE:           {test_dice:.6f} (± {np.std(dice_scores):.6f})\n")
    f.write(f"  Final Hausdorff_95:   {test_hausdorff_95:.6f}px (± {np.std(hausdorff_95_scores):.6f}px)\n")
    f.write(f"  Final Blockwise Diff: {overall_avg_blockwise_diff:.6f} (± {np.std(all_blockwise_diffs):.6f})\n")
